## Create BGEN manifest 

Here we split ACAF and exome_only bgen files into chunks of 200 000.

This assumes we have already run nextflow_exome_specific.ipynb.

In [43]:
%%bash
{
    printf 'bgen,basename\n'
    gcloud storage ls 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/*.bgen' \
      --billing-project $GOOGLE_PROJECT \
    | awk -F/ '{ print $0 "," $NF }'
  } > acaf_bgen.csv


In [48]:
! head acaf_bgen.csv

bgen,basename
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr1.bgen,acaf_threshold.chr1.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr10.bgen,acaf_threshold.chr10.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr11.bgen,acaf_threshold.chr11.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr12.bgen,acaf_threshold.chr12.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr13.bgen,acaf_threshold.chr13.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr14.bgen,acaf_threshold.chr14.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr15.bgen,acaf_threshold.chr15.bgen
gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf

In [42]:
%%bash

echo "basename,chrom,start,end" > acaf_chunks.csv

for chr in {1..22} X; do
    bgen_filename="/home/jupyter/workspace/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/bgen/acaf_threshold.chr${chr}"
    bgen_basename=$(basename $bgen_filename)
    sqlite3 -csv ${bgen_filename}.bgen.bgi "
        WITH variant_ordered AS (
        SELECT chromosome, position,
           ROW_NUMBER() OVER (PARTITION BY chromosome ORDER BY position) AS row_num
        FROM Variant
        ),
        chunked AS (
        SELECT 
           chromosome,
           MIN(position) AS startpos,
           MAX(position) AS endpos
        FROM variant_ordered
        GROUP BY chromosome, (row_num - 1) / 200000
        )
        SELECT '${bgen_basename}.bgen' as basename,
          chromosome, 
          startpos, 
          endpos 
        FROM chunked 
        ORDER BY chromosome, startpos;" >> acaf_chunks.csv
done


In [50]:
! head acaf_chunks.csv

basename,chrom,start,end
acaf_threshold.chr1.bgen,chr1,10060,2927788
acaf_threshold.chr1.bgen,chr1,2927788,6364015
acaf_threshold.chr1.bgen,chr1,6364025,10226829
acaf_threshold.chr1.bgen,chr1,10226829,14209824
acaf_threshold.chr1.bgen,chr1,14209872,18244545
acaf_threshold.chr1.bgen,chr1,18244547,22375035
acaf_threshold.chr1.bgen,chr1,22375035,26400476
acaf_threshold.chr1.bgen,chr1,26400539,30780009
acaf_threshold.chr1.bgen,chr1,30780009,35220562


In [55]:
%%bash
{
    printf 'bgen,basename\n'
    gcloud storage ls 'gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/*.bgen' \
      --billing-project $GOOGLE_PROJECT \
    | awk -F/ '{ print $0 "," $NF }'
  } > exome_only_bgen.csv


In [56]:
! head exome_only_bgen.csv

bgen,basename
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr10_exome_only.bgen,chr10_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr11_exome_only.bgen,chr11_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr12_exome_only.bgen,chr12_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr13_exome_only.bgen,chr13_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr14_exome_only.bgen,chr14_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr15_exome_only.bgen,chr15_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr16_exome_only.bgen,chr16_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr17_exome_only.bgen,chr17_exome_only.bgen
gs://workspace-bucket-wb-radiant-cabbage-3726/exome_only/chr18_exome_only.bgen,chr18_exome_only.bgen


In [46]:
%%bash

echo "basename,chrom,start,end" > exome_only_chunks.csv

for chr in {1..22} X; do
    bgen_filename="/home/jupyter/workspace/workspace-bucket/exome_only/chr${chr}_exome_only"
    bgen_basename=$(basename $bgen_filename)
    sqlite3 -csv ${bgen_filename}.bgen.bgi "
        WITH variant_ordered AS (
        SELECT chromosome, position,
           ROW_NUMBER() OVER (PARTITION BY chromosome ORDER BY position) AS row_num
        FROM Variant
        ),
        chunked AS (
        SELECT 
           chromosome,
           MIN(position) AS startpos,
           MAX(position) AS endpos
        FROM variant_ordered
        GROUP BY chromosome, (row_num - 1) / 200000
        )
        SELECT '${bgen_basename}.bgen' as bgen,
          chromosome, 
          startpos, 
          endpos 
        FROM chunked 
        ORDER BY chromosome, startpos;" >> exome_only_chunks.csv
done


In [49]:
! head exome_only_chunks.csv

basename,chrom,start,end
chr1_exome_only.bgen,chr1,65529,6294955
chr1_exome_only.bgen,chr1,6294958,13370877
chr1_exome_only.bgen,chr1,13370883,21622461
chr1_exome_only.bgen,chr1,21622462,27296408
chr1_exome_only.bgen,chr1,27296409,34988588
chr1_exome_only.bgen,chr1,34988589,42188933
chr1_exome_only.bgen,chr1,42188936,50438565
chr1_exome_only.bgen,chr1,50438567,64179449
chr1_exome_only.bgen,chr1,64179450,84645127


In [57]:
import pandas as pd

acaf_bgen = pd.read_csv("acaf_bgen.csv")
acaf_chunks = pd.read_csv("acaf_chunks.csv")
exome_bgen = pd.read_csv("exome_only_bgen.csv")
exome_chunks = pd.read_csv("exome_only_chunks.csv")

In [71]:
manifest = pd.concat([pd.merge(acaf_bgen, acaf_chunks, on='basename').assign(source="acaf"),
           pd.merge(exome_bgen, exome_chunks, on='basename').assign(source="exome_specific")])

manifest


,bgen,basename,chrom,start,end,source
0,gs://vwb-aou-datasets-controlled/v9/wgs/short_...,acaf_threshold.chr1.bgen,chr1,10060,2927788,acaf
1,gs://vwb-aou-datasets-controlled/v9/wgs/short_...,acaf_threshold.chr1.bgen,chr1,2927788,6364015,acaf
2,gs://vwb-aou-datasets-controlled/v9/wgs/short_...,acaf_threshold.chr1.bgen,chr1,6364025,10226829,acaf
3,gs://vwb-aou-datasets-controlled/v9/wgs/short_...,acaf_threshold.chr1.bgen,chr1,10226829,14209824,acaf
4,gs://vwb-aou-datasets-controlled/v9/wgs/short_...,acaf_threshold.chr1.bgen,chr1,14209872,18244545,acaf
...,...,...,...,...,...,...
231,gs://workspace-bucket-wb-radiant-cabbage-3726/...,chrX_exome_only.bgen,chrX,49174391,72148441,exome_specific
232,gs://workspace-bucket-wb-radiant-cabbage-3726/...,chrX_exome_only.bgen,chrX,72148443,107600562,exome_specific
233,gs://workspace-bucket-wb-radiant-cabbage-3726/...,chrX_exome_only.bgen,chrX,107600568,134413557,exome_specific
234,gs://workspace-bucket-wb-radiant-cabbage-3726/...,chrX_exome_only.bgen,chrX,134413559,154411597,exome_specific


In [72]:
manifest["id"] = (manifest.source + "_" + 
     manifest.chrom + "_" + 
     manifest.start.astype(str) + "_" +
     manifest.end.astype(str))

In [76]:
manifest.get(['bgen', 'id', 'chrom', 'start', 'end']).to_csv("manifest.tsv", sep="\t", index=False)